# RecursiveJsonSplitter

JSON 데이터를 깊이 우선 탐색(depth-first traversal)하여 더 작은 JSON 청크를 만드는 분할기입니다.

중첩된 JSON 객체를 가능한 한 유지하려고 하지만, 청크 크기를 `min_chunk_size` 와 `max_chunk_size` 사이로 맞추기 위해 필요하면 객체를 분할합니다.
값이 중첩 JSON이 아니라 매우 긴 **문자열**이면 그 문자열은 분할되지 않습니다.
청크 크기에 엄격한 상한이 필요하면 이 분할기 뒤에 `RecursiveCharacterTextSplitter` 를 이어 붙이세요.

**분할 기준**
1. 텍스트 분할 방식: JSON 값 기준
2. 청크 크기 측정 방식: 문자 수 기준 (`len(json.dumps(chunk))`)

> **🔄 최신 버전 기준 변경 사항 (langchain-text-splitters 1.x)**
> - HTTP 요청에 `httpx` 를 사용하고, **타임아웃과 상태 코드 검사**를 추가했습니다.
> - `ensure_ascii=False` 옵션으로 한글 등 비ASCII 문자를 `\uXXXX` 이스케이프 없이 출력하는 방법을 추가했습니다.
> - **원본 코드의 버그 수정**
>   - `json_data = json.loads(texts[2])` 가 원본 데이터를 **덮어써서**, 이후 `convert_lists=True` 예제가 전체 데이터가 아닌 청크 하나에 대해 실행되었습니다. 변수명을 분리했습니다.
>   - 설명(`texts[2]`)과 코드(`texts[1]`)의 인덱스가 서로 달랐습니다. 크기 제한을 넘는 청크를 **코드로 찾아서** 확인하도록 바꿨습니다.
> - 크기 상한을 엄격히 지키기 위한 2단계 분할 예제를 추가했습니다.

In [ ]:
%pip install -qU langchain-text-splitters httpx

LangSmith API의 OpenAPI 명세(JSON)를 가져와 파이썬 딕셔너리로 변환합니다.

In [ ]:
import httpx

response = httpx.get("https://api.smith.langchain.com/openapi.json", timeout=30.0)
response.raise_for_status()  # 🔄 HTTP 오류(4xx/5xx)면 예외 발생
json_data = response.json()

In [ ]:
# 전체 출력은 매우 길기 때문에 최상위 키만 확인합니다.
list(json_data.keys())

`RecursiveJsonSplitter` 를 생성합니다. `min_chunk_size` 를 생략하면 `max_chunk_size - 200` (최소 50)으로 설정됩니다.

In [ ]:
from langchain_text_splitters import RecursiveJsonSplitter

splitter = RecursiveJsonSplitter(max_chunk_size=300)

`split_json()` 은 **딕셔너리 리스트**를 반환합니다. 작은 JSON 조각을 파이썬 객체로 다뤄야 할 때 사용합니다.

In [ ]:
json_chunks = splitter.split_json(json_data=json_data)

for chunk in json_chunks[:3]:
    print(chunk)

- `create_documents()`: JSON을 `Document` 리스트로 변환
- `split_text()`: JSON을 **JSON 문자열 리스트**로 변환
- 🔄 `ensure_ascii=False`: 한글 등이 `\uXXXX` 로 이스케이프되지 않고 그대로 출력됩니다. (기본값은 `True`)

In [ ]:
docs = splitter.create_documents(texts=[json_data], ensure_ascii=False)
texts = splitter.split_text(json_data=json_data, ensure_ascii=False)

print(docs[0].page_content)
print("===" * 20)
print(texts[0])

## 크기 제한을 넘는 청크 확인

각 청크의 크기를 확인해 보면 `max_chunk_size=300` 을 넘는 청크가 있습니다.
이는 `RecursiveJsonSplitter` 가 **리스트(list) 객체는 분할하지 않기 때문**입니다.

🔄 인덱스를 추측하지 않고, 제한을 넘는 청크를 코드로 찾아서 확인합니다.

In [ ]:
sizes = [len(text) for text in texts]
print("앞 10개 청크 크기:", sizes[:10])

oversized = [i for i, size in enumerate(sizes) if size > 300]
print(f"300자를 넘는 청크: {len(oversized)}개 / 전체 {len(texts)}개")

big_idx = oversized[0]
print(f"\n[{big_idx}번 청크, {sizes[big_idx]}자]")
print(texts[big_idx])

청크 문자열은 `json` 모듈로 다시 파싱할 수 있습니다.

🔄 원본은 이 결과를 `json_data` 에 저장해 원본 데이터를 덮어썼습니다. 별도의 변수(`big_chunk`)에 저장합니다.

In [ ]:
import json

big_chunk = json.loads(texts[big_idx])
big_chunk

## 리스트를 딕셔너리로 변환하여 분할 (`convert_lists=True`)

`convert_lists=True` 로 설정하면 전처리 단계에서 JSON 안의 리스트를 `{"0": item0, "1": item1, ...}` 형태의 딕셔너리로 바꾼 뒤 분할합니다.
그러면 리스트 내부도 분할할 수 있어 크기 제한을 넘는 청크가 줄어듭니다.

In [ ]:
# 🔄 덮어쓰이지 않은 원본 json_data 전체에 대해 실행합니다.
converted_texts = splitter.split_text(
    json_data=json_data, convert_lists=True, ensure_ascii=False
)

converted_sizes = [len(t) for t in converted_texts]
print(f"convert_lists=False → 청크 {len(texts)}개, 최대 {max(sizes)}자")
print(f"convert_lists=True  → 청크 {len(converted_texts)}개, 최대 {max(converted_sizes)}자")

In [ ]:
# 리스트가 딕셔너리로 변환된 결과를 확인합니다.
print(converted_texts[2])

`docs` 리스트의 특정 문서를 확인할 수 있습니다.

In [ ]:
docs[2]

## 🔄 추가: 엄격한 크기 상한이 필요할 때 (2단계 분할)

리스트를 변환해도 **아주 긴 문자열 값**은 쪼개지지 않으므로 여전히 제한을 넘을 수 있습니다.
임베딩 모델의 입력 한도처럼 반드시 지켜야 하는 상한이 있다면 `RecursiveCharacterTextSplitter` 로 한 번 더 나눕니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

json_docs = splitter.create_documents(
    texts=[json_data], convert_lists=True, ensure_ascii=False
)

char_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)
final_docs = char_splitter.split_documents(json_docs)

print(f"JSON 분할: {len(json_docs)}개 → 최종: {len(final_docs)}개")
print("최종 최대 길이:", max(len(d.page_content) for d in final_docs))

> 참고: `RecursiveJsonSplitter` 는 크기를 잴 때 내부적으로 `json.dumps()` 기본값(`ensure_ascii=True`)을 사용합니다.
> 그래서 한글이 많은 JSON은 실제 출력 문자열보다 **더 크게 측정**(한 글자 = `\uXXXX` 6자)되어 청크가 예상보다 작게 나뉠 수 있습니다.